## 1.load the dataset

In [2]:
import pandas as pd

df=pd.read_csv("childmortalitydataset.csv")   # to load dataset

## 2.understand the data

In [3]:
df.head()    # first 5 rows

,Record_ID,Country,Region,Child_Age_Months,Gender,Birth_Weight_kg,Mother_Age,Immunization_Status,Nutrition_Status,Cause_of_Death,Death_Date
0,1,India,Rural,44,Male,1.52,31,Partially Immunized,Severe Malnutrition,Prematurity,2023-11-28
1,2,Kenya,Rural,29,Male,2.91,27,Not Immunized,Moderate Malnutrition,Malaria,2023-03-12
2,3,Brazil,Rural,38,Female,3.35,18,Not Immunized,Normal,Pneumonia,2023-04-13
3,4,Brazil,Rural,40,Female,3.14,32,Not Immunized,Severe Malnutrition,Prematurity,2023-08-17
4,5,Bangladesh,Rural,48,Male,2.79,21,Fully Immunized,Normal,Pneumonia,2023-11-11


In [4]:
df.tail()    # to show last 5 rows

,Record_ID,Country,Region,Child_Age_Months,Gender,Birth_Weight_kg,Mother_Age,Immunization_Status,Nutrition_Status,Cause_of_Death,Death_Date
295,296,Brazil,Urban,15,Female,3.11,24,Partially Immunized,Normal,Other,2023-02-03
296,297,India,Urban,59,Female,3.35,20,Partially Immunized,Severe Malnutrition,Prematurity,2023-08-07
297,298,Nigeria,Rural,32,Male,1.99,26,Fully Immunized,Normal,Other,2023-10-24
298,299,India,Rural,40,Female,1.78,35,Fully Immunized,Moderate Malnutrition,Diarrhea,2023-02-17
299,300,Nigeria,Rural,52,Male,2.75,19,Not Immunized,Severe Malnutrition,Other,2023-03-20


In [5]:
df.info()     # data types + missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Record_ID            300 non-null    int64  
 1   Country              300 non-null    object 
 2   Region               300 non-null    object 
 3   Child_Age_Months     300 non-null    int64  
 4   Gender               300 non-null    object 
 5   Birth_Weight_kg      300 non-null    float64
 6   Mother_Age           300 non-null    int64  
 7   Immunization_Status  300 non-null    object 
 8   Nutrition_Status     300 non-null    object 
 9   Cause_of_Death       300 non-null    object 
 10  Death_Date           300 non-null    object 
dtypes: float64(1), int64(3), object(7)
memory usage: 25.9+ KB


In [6]:
df.describe()    # stats for numeric columns

,Record_ID,Child_Age_Months,Birth_Weight_kg,Mother_Age
count,300.000000,300.000000,300.000000,300.000000
mean,150.500000,30.556667,2.808733,28.783333
std,86.746758,17.686043,0.709687,6.734555
min,1.000000,0.000000,1.500000,18.000000
25%,75.750000,16.000000,2.197500,23.000000
50%,150.500000,30.000000,2.825000,29.000000
75%,225.250000,47.000000,3.422500,34.000000
max,300.000000,59.000000,3.980000,40.000000


In [7]:
df.shape   # to display row and column numbers

(300, 11)

## 3.standardise column names

In [8]:
df.columns=df.columns.str.strip().str.lower().str.replace(" ", "_")    # to make column names standardize

## 4.check & handle missing values

In [9]:
df.isnull().sum()    # count missing values

record_id              0
country                0
region                 0
child_age_months       0
gender                 0
birth_weight_kg        0
mother_age             0
immunization_status    0
nutrition_status       0
cause_of_death         0
death_date             0
dtype: int64

## there is no missing values so don't want to do drop or fill

## 5.remove duplicates

In [10]:
df.duplicated()   # to check duplicates 

0      False
1      False
2      False
3      False
4      False
       ...  
295    False
296    False
297    False
298    False
299    False
Length: 300, dtype: bool

In [11]:
df.duplicated().sum()   # to count duplicate

np.int64(0)

## there is no duplicates so don't want to drop duplicates

## 6.convert data types

In [12]:
df['death_date']=pd.to_datetime(df['death_date'],errors='coerce')    # converts the  datatype string to datetime of the column death date if value cannot be convertedit replaces with NAT(not a time) 

## 7.handle invalid dates

In [13]:
df=df.dropna(subset=['death_date'])    # it removes rows where death_date is missing

## 8.clean text data

In [14]:
cols=['country','region','gender','immunization_status','nutrition_status','cause_of_death']

for col in cols:
    df[col]=df[col].str.strip().str.lower()      # removes spaces and converts into lower case

## 9.fix inconsistent value

In [15]:
df['gender']=df['gender'].replace({'m':'male','f':'female'})   # changes gender valeu male to 'm' and female to 'f'.

## 10.validate data ranges

In [16]:
# removing unrealistic values

# Age (0–60 months)
df = df[(df['child_age_months'] >= 0) & (df['child_age_months'] <= 60)]

# Birth weight (1–5 kg)
df = df[(df['birth_weight_kg'] >= 1) & (df['birth_weight_kg'] <= 5)]

# Mother age (15–50)
df = df[(df['mother_age'] >= 15) & (df['mother_age'] <= 50)]

## 11.detect and handle outliers

In [17]:
# handling outliers using IQR method

q1 = df['birth_weight_kg'].quantile(0.25)
q3 = df['birth_weight_kg'].quantile(0.75)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df['birth_weight_kg'] = df['birth_weight_kg'].clip(lower, upper)

## 12.feature engineering(optional but important)

In [18]:
df['year'] = df['death_date'].dt.year      # extracting year from death date
df['month'] = df['death_date'].dt.month_name()    # extracting monthname from death date

df['age_group'] = pd.cut(df['child_age_months'],
                        bins=[0, 12, 36, 60],
                        labels=['infant', 'toddler', 'child'])

In [19]:
print(df)

     record_id     country region  child_age_months  gender  birth_weight_kg  \
0            1       india  rural                44    male             1.52   
1            2       kenya  rural                29    male             2.91   
2            3      brazil  rural                38  female             3.35   
3            4      brazil  rural                40  female             3.14   
4            5  bangladesh  rural                48    male             2.79   
..         ...         ...    ...               ...     ...              ...   
295        296      brazil  urban                15  female             3.11   
296        297       india  urban                59  female             3.35   
297        298     nigeria  rural                32    male             1.99   
298        299       india  rural                40  female             1.78   
299        300     nigeria  rural                52    male             2.75   

     mother_age  immunization_status   

## 13.final check

In [20]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   record_id            300 non-null    int64         
 1   country              300 non-null    object        
 2   region               300 non-null    object        
 3   child_age_months     300 non-null    int64         
 4   gender               300 non-null    object        
 5   birth_weight_kg      300 non-null    float64       
 6   mother_age           300 non-null    int64         
 7   immunization_status  300 non-null    object        
 8   nutrition_status     300 non-null    object        
 9   cause_of_death       300 non-null    object        
 10  death_date           300 non-null    datetime64[ns]
 11  year                 300 non-null    int32         
 12  month                300 non-null    object        
 13  age_group            295 non-null  

record_id              0
country                0
region                 0
child_age_months       0
gender                 0
birth_weight_kg        0
mother_age             0
immunization_status    0
nutrition_status       0
cause_of_death         0
death_date             0
year                   0
month                  0
age_group              5
dtype: int64

In [21]:
df[df['age_group'].isnull()]  # to show which rows have issue

,record_id,country,region,child_age_months,gender,birth_weight_kg,mother_age,immunization_status,nutrition_status,cause_of_death,death_date,year,month,age_group
5,6,kenya,urban,0,male,2.36,29,fully immunized,normal,other,2023-10-10,2023,October,NaN
108,109,india,rural,0,female,2.61,18,fully immunized,severe malnutrition,pneumonia,2023-04-04,2023,April,NaN
168,169,nigeria,rural,0,female,2.16,20,not immunized,severe malnutrition,malaria,2023-06-11,2023,June,NaN
204,205,kenya,rural,0,female,2.08,30,not immunized,severe malnutrition,diarrhea,2023-10-13,2023,October,NaN
283,284,kenya,rural,0,male,2.66,26,fully immunized,normal,prematurity,2023-05-09,2023,May,NaN


In [22]:
# the problem behind is when before i declared child age group 0-12 months is infant so in this 0 not included.so the children who died at 0 month age will not declared age group so want to fix that. 
# to fix issue

df['age_group'] = pd.cut(df['child_age_months'],
                        bins=[-1, 12, 36, 60],
                        labels=['infant', 'toddler', 'child'])

In [23]:
df.isnull().sum()

record_id              0
country                0
region                 0
child_age_months       0
gender                 0
birth_weight_kg        0
mother_age             0
immunization_status    0
nutrition_status       0
cause_of_death         0
death_date             0
year                   0
month                  0
age_group              0
dtype: int64

In [24]:
df.to_csv("cleaned_child_mortality.csv", index=False)